In [4]:
"""
open_cluster_simulator.py

Simulador cinemático tipo Gaia para cúmulos abiertos.

El modelo genera miembros de un cúmulo abierto en 3D, asigna una
velocidad espacial común dirigida hacia un ápex galáctico y proyecta
esa cinemática a observables astrométricos tipo Gaia:

    ra        [deg]
    dec       [deg]
    parallax  [mas]
    pmra      [mas / yr]  = mu_alpha * cos(dec)
    pmdec     [mas / yr]

Este modelo no incluye todavía fotometría ni evolución N-body. Es un
modelo cinemático diseñado para probar métodos de ápex, convergencia y
clustering en espacio astrométrico.
"""

from __future__ import annotations

from dataclasses import dataclass
from typing import Optional

import numpy as np
import pandas as pd
from astropy import units as u
from astropy.coordinates import SkyCoord


KM_S_PER_ARCSEC_YR_PC = 4.74047


@dataclass(frozen=True)
class ClusterSimulationConfig:
    """
    Configuración física y observacional de la simulación.

    Parameters
    ----------
    n_members : int
        Número de estrellas simuladas.

    center_ra_deg : float
        Ascensión recta del centro del cúmulo, en grados.

    center_dec_deg : float
        Declinación del centro del cúmulo, en grados.

    distance_pc : float
        Distancia heliocéntrica al centro del cúmulo, en parsec.

    radius_pc : float
        Radio físico máximo del cúmulo, en parsec.

    apex_l_deg : float
        Longitud galáctica del ápex, en grados.

    apex_b_deg : float
        Latitud galáctica del ápex, en grados.

    speed_kms : float
        Velocidad espacial total del cúmulo, en km/s.

    speed_sigma_kms : float, optional
        Dispersión gaussiana estrella-a-estrella en velocidad, en km/s.

    apex_l_sigma_deg : float, optional
        Dispersión gaussiana en longitud galáctica del ápex, en grados.

    apex_b_sigma_deg : float, optional
        Dispersión gaussiana en latitud galáctica del ápex, en grados.

    parallax_error_mas : float, optional
        Error observacional gaussiano aplicado a la paralaje, en mas.

    proper_motion_error_masyr : float, optional
        Error observacional gaussiano aplicado a pmra y pmdec, en mas/yr.

    position_error_mas : float, optional
        Error observacional gaussiano aplicado a ra y dec, en mas.

    seed : int or None, optional
        Semilla para reproducibilidad.

    include_true_values : bool, optional
        Si True, agrega columnas con los valores sin ruido observacional.
    """

    n_members: int
    center_ra_deg: float
    center_dec_deg: float
    distance_pc: float
    radius_pc: float
    apex_l_deg: float
    apex_b_deg: float
    speed_kms: float
    speed_sigma_kms: float = 0.0
    apex_l_sigma_deg: float = 0.0
    apex_b_sigma_deg: float = 0.0
    parallax_error_mas: float = 0.0
    proper_motion_error_masyr: float = 0.0
    position_error_mas: float = 0.0
    seed: Optional[int] = None
    include_true_values: bool = True


class OpenClusterSimulator:
    """
    Simulador cinemático de cúmulos abiertos tipo Gaia.
    """

    def __init__(self, config: ClusterSimulationConfig) -> None:
        """
        Inicializa el simulador.

        Parameters
        ----------
        config : ClusterSimulationConfig
            Configuración física y observacional del cúmulo.
        """

        self.config = config
        self._validate_config()
        self.rng = np.random.default_rng(config.seed)

    def simulate(self) -> pd.DataFrame:
        """
        Ejecuta la simulación completa.

        Returns
        -------
        pd.DataFrame
            Catálogo simulado con columnas principales:

            - source_id
            - ra
            - dec
            - parallax
            - pmra
            - pmdec

            También incluye columnas verdaderas y diagnósticas.
        """

        positions_pc = self._build_cluster_positions()
        velocities_kms = self._build_cluster_velocities()

        catalog = self._phase_space_to_gaia_observables(
            positions_pc=positions_pc,
            velocities_kms=velocities_kms,
        )

        catalog.insert(
            loc=0,
            column="source_id",
            value=np.arange(1, self.config.n_members + 1),
        )

        if self.config.include_true_values:
            catalog["ra_true"] = catalog["ra"]
            catalog["dec_true"] = catalog["dec"]
            catalog["parallax_true"] = catalog["parallax"]
            catalog["pmra_true"] = catalog["pmra"]
            catalog["pmdec_true"] = catalog["pmdec"]

        catalog = self._apply_observational_noise(catalog)

        return catalog

    def _validate_config(self) -> None:
        """
        Valida que los parámetros sean físicamente razonables.
        """

        config = self.config

        if config.n_members <= 0:
            raise ValueError("n_members must be greater than zero.")

        if config.distance_pc <= 0.0:
            raise ValueError("distance_pc must be greater than zero.")

        if config.radius_pc < 0.0:
            raise ValueError("radius_pc must be non-negative.")

        if config.radius_pc >= config.distance_pc:
            raise ValueError(
                "radius_pc must be smaller than distance_pc. "
                "Otherwise some sources may have non-physical distances."
            )

        if config.speed_kms < 0.0:
            raise ValueError("speed_kms must be non-negative.")

        sigma_values = {
            "speed_sigma_kms": config.speed_sigma_kms,
            "apex_l_sigma_deg": config.apex_l_sigma_deg,
            "apex_b_sigma_deg": config.apex_b_sigma_deg,
            "parallax_error_mas": config.parallax_error_mas,
            "proper_motion_error_masyr": (
                config.proper_motion_error_masyr
            ),
            "position_error_mas": config.position_error_mas,
        }

        for name, value in sigma_values.items():
            if value < 0.0:
                raise ValueError(f"{name} must be non-negative.")

    def _build_cluster_positions(self) -> np.ndarray:
        """
        Genera posiciones cartesianas heliocéntricas ICRS en pc.

        Returns
        -------
        np.ndarray
            Arreglo de forma ``(n_members, 3)`` con posiciones en pc.
        """

        config = self.config

        center_coord = SkyCoord(
            ra=config.center_ra_deg * u.deg,
            dec=config.center_dec_deg * u.deg,
            distance=config.distance_pc * u.pc,
            frame="icrs",
        )

        center_xyz_pc = center_coord.cartesian.xyz.to_value(u.pc)

        offsets_pc = self._sample_uniform_sphere(
            n_points=config.n_members,
            radius_pc=config.radius_pc,
        )

        return center_xyz_pc[None, :] + offsets_pc

    def _build_cluster_velocities(self) -> np.ndarray:
        """
        Genera velocidades cartesianas heliocéntricas ICRS en km/s.

        Returns
        -------
        np.ndarray
            Arreglo de forma ``(n_members, 3)`` con velocidades en km/s.
        """

        config = self.config

        speeds_kms = self.rng.normal(
            loc=config.speed_kms,
            scale=config.speed_sigma_kms,
            size=config.n_members,
        )
        speeds_kms = np.clip(speeds_kms, a_min=0.0, a_max=None)

        apex_l_deg = self.rng.normal(
            loc=config.apex_l_deg,
            scale=config.apex_l_sigma_deg,
            size=config.n_members,
        )
        apex_l_deg = np.mod(apex_l_deg, 360.0)

        apex_b_deg = self.rng.normal(
            loc=config.apex_b_deg,
            scale=config.apex_b_sigma_deg,
            size=config.n_members,
        )
        apex_b_deg = np.clip(apex_b_deg, -90.0, 90.0)

        apex_unit_vectors = self._galactic_apex_to_icrs_unit_vector(
            apex_l_deg=apex_l_deg,
            apex_b_deg=apex_b_deg,
        )

        return speeds_kms[:, None] * apex_unit_vectors

    def _sample_uniform_sphere(
        self,
        n_points: int,
        radius_pc: float,
    ) -> np.ndarray:
        """
        Muestrea puntos uniformemente dentro de una esfera 3D.

        Parameters
        ----------
        n_points : int
            Número de puntos.

        radius_pc : float
            Radio máximo de la esfera, en pc.

        Returns
        -------
        np.ndarray
            Offsets cartesianos de forma ``(n_points, 3)`` en pc.
        """

        if radius_pc == 0.0:
            return np.zeros((n_points, 3), dtype=float)

        directions = self.rng.normal(size=(n_points, 3))
        norms = np.linalg.norm(directions, axis=1)

        if np.any(norms == 0.0):
            raise RuntimeError("Random direction with zero norm generated.")

        unit_directions = directions / norms[:, None]
        radii = radius_pc * self.rng.random(n_points) ** (1.0 / 3.0)

        return unit_directions * radii[:, None]

    @staticmethod
    def _galactic_apex_to_icrs_unit_vector(
        apex_l_deg: np.ndarray,
        apex_b_deg: np.ndarray,
    ) -> np.ndarray:
        """
        Convierte direcciones de ápex galácticas a vectores ICRS.

        Parameters
        ----------
        apex_l_deg : np.ndarray
            Longitudes galácticas, en grados.

        apex_b_deg : np.ndarray
            Latitudes galácticas, en grados.

        Returns
        -------
        np.ndarray
            Vectores unitarios ICRS de forma ``(n_sources, 3)``.
        """

        apex_coords = SkyCoord(
            l=apex_l_deg * u.deg,
            b=apex_b_deg * u.deg,
            frame="galactic",
        )

        icrs_cartesian = apex_coords.icrs.cartesian

        return np.column_stack(
            (
                icrs_cartesian.x.value,
                icrs_cartesian.y.value,
                icrs_cartesian.z.value,
            )
        )

    @staticmethod
    def _phase_space_to_gaia_observables(
        positions_pc: np.ndarray,
        velocities_kms: np.ndarray,
    ) -> pd.DataFrame:
        """
        Convierte fase espacial cartesiana ICRS a observables tipo Gaia.

        Parameters
        ----------
        positions_pc : np.ndarray
            Posiciones heliocéntricas cartesianas ICRS, en pc.
            Forma esperada: ``(n_sources, 3)``.

        velocities_kms : np.ndarray
            Velocidades heliocéntricas cartesianas ICRS, en km/s.
            Forma esperada: ``(n_sources, 3)``.

        Returns
        -------
        pd.DataFrame
            DataFrame con ra, dec, parallax, pmra y pmdec.
        """

        if positions_pc.shape != velocities_kms.shape:
            raise ValueError(
                "positions_pc and velocities_kms must have the same shape."
            )

        if positions_pc.ndim != 2 or positions_pc.shape[1] != 3:
            raise ValueError(
                "positions_pc and velocities_kms must have shape "
                "(n_sources, 3)."
            )

        x_coord = positions_pc[:, 0]
        y_coord = positions_pc[:, 1]
        z_coord = positions_pc[:, 2]

        distance_pc = np.linalg.norm(positions_pc, axis=1)

        if np.any(distance_pc <= 0.0):
            raise ValueError("All sources must have positive distance.")

        unit_position = positions_pc / distance_pc[:, None]

        ra_rad = np.arctan2(y_coord, x_coord) % (2.0 * np.pi)
        dec_rad = np.arcsin(np.clip(z_coord / distance_pc, -1.0, 1.0))

        sin_ra = np.sin(ra_rad)
        cos_ra = np.cos(ra_rad)
        sin_dec = np.sin(dec_rad)
        cos_dec = np.cos(dec_rad)

        basis_ra = np.column_stack(
            (
                -sin_ra,
                cos_ra,
                np.zeros_like(ra_rad),
            )
        )

        basis_dec = np.column_stack(
            (
                -cos_ra * sin_dec,
                -sin_ra * sin_dec,
                cos_dec,
            )
        )

        velocity_ra_kms = np.sum(velocities_kms * basis_ra, axis=1)
        velocity_dec_kms = np.sum(velocities_kms * basis_dec, axis=1)

        radial_velocity_kms = np.sum(
            velocities_kms * unit_position,
            axis=1,
        )

        pmra_masyr = (
            1_000.0
            * velocity_ra_kms
            / (KM_S_PER_ARCSEC_YR_PC * distance_pc)
        )

        pmdec_masyr = (
            1_000.0
            * velocity_dec_kms
            / (KM_S_PER_ARCSEC_YR_PC * distance_pc)
        )

        return pd.DataFrame(
            {
                "ra": np.degrees(ra_rad),
                "dec": np.degrees(dec_rad),
                "parallax": 1_000.0 / distance_pc,
                "pmra": pmra_masyr,
                "pmdec": pmdec_masyr,
                "distance_pc_true": distance_pc,
                "radial_velocity_kms_true": radial_velocity_kms,
            }
        )

    def _apply_observational_noise(
        self,
        catalog: pd.DataFrame,
    ) -> pd.DataFrame:
        """
        Aplica ruido observacional gaussiano a las columnas tipo Gaia.

        Parameters
        ----------
        catalog : pd.DataFrame
            Catálogo sin ruido observacional.

        Returns
        -------
        pd.DataFrame
            Catálogo con ruido observacional.
        """

        config = self.config
        result = catalog.copy()

        if config.position_error_mas > 0.0:
            sigma_deg = config.position_error_mas / 3_600_000.0

            result["ra"] += self.rng.normal(
                loc=0.0,
                scale=sigma_deg,
                size=len(result),
            )
            result["dec"] += self.rng.normal(
                loc=0.0,
                scale=sigma_deg,
                size=len(result),
            )

            result["ra"] = np.mod(result["ra"], 360.0)
            result["dec"] = np.clip(result["dec"], -90.0, 90.0)

        if config.parallax_error_mas > 0.0:
            result["parallax"] += self.rng.normal(
                loc=0.0,
                scale=config.parallax_error_mas,
                size=len(result),
            )

        if config.proper_motion_error_masyr > 0.0:
            result["pmra"] += self.rng.normal(
                loc=0.0,
                scale=config.proper_motion_error_masyr,
                size=len(result),
            )
            result["pmdec"] += self.rng.normal(
                loc=0.0,
                scale=config.proper_motion_error_masyr,
                size=len(result),
            )

        return result


if __name__ == "__main__":
    config = ClusterSimulationConfig(
        n_members=300,
        center_ra_deg=66,
        center_dec_deg=15,
        distance_pc=85.0,
        radius_pc=5.0,
        apex_l_deg=202.51,
        apex_b_deg=3.16,
        speed_kms=25.0,
        speed_sigma_kms=0.3,
        apex_l_sigma_deg=0.5,
        apex_b_sigma_deg=0.5,
        parallax_error_mas=0.02,
        proper_motion_error_masyr=0.05,
        position_error_mas=0.1,
        seed=42,
    )

    simulator = OpenClusterSimulator(config)
    catalog = simulator.simulate()

    print(catalog.head())
    catalog.to_csv("mock_open_cluster_gaia_like.csv", index=False)

   source_id         ra        dec   parallax       pmra     pmdec  \
0          1  64.586960  16.806201  12.004526  36.181656 -3.398525   
1          2  63.852382  13.860347  12.194985  37.420683  0.002147   
2          3  63.608126  15.411051  12.212460  39.030815 -2.405756   
3          4  67.598122  15.850499  11.563652  32.071664 -2.363749   
4          5  67.059392  15.454266  11.178491  31.625147 -1.968088   

   distance_pc_true  radial_velocity_kms_true    ra_true   dec_true  \
0         83.370706                 19.993140  64.586960  16.806201   
1         82.020513                 20.020006  63.852382  13.860347   
2         81.936352                 20.427062  63.608126  15.411051   
3         86.349140                 20.874830  67.598122  15.850499   
4         89.594923                 20.613027  67.059392  15.454266   

   parallax_true  pmra_true  pmdec_true  
0      11.994621  36.242159   -3.420636  
1      12.192072  37.304132    0.005193  
2      12.204595  38.98984

In [ ]:
# ejemplo de uso
# from open_cluster_simulator import (
#     ClusterSimulationConfig,
#     OpenClusterSimulator,
# )

# config = ClusterSimulationConfig(
#     n_members=500,
#     center_ra_deg=186.25,
#     center_dec_deg=26.10,
#     distance_pc=85.0,
#     radius_pc=5.0,
#     apex_l_deg=180.0,
#     apex_b_deg=0.0,
#     speed_kms=25.0,
#     speed_sigma_kms=0.3,
#     apex_l_sigma_deg=1.0,
#     apex_b_sigma_deg=0.5,
#     parallax_error_mas=0.02,
#     proper_motion_error_masyr=0.05,
#     position_error_mas=0.1,
#     seed=42,
# )

# simulator = OpenClusterSimulator(config)
# df_cluster = simulator.simulate()

# df_cluster.head()

NameError: name 'ClusterSimulationConfig' is not defined